# Worker ADACEEN sobre Colab

Corre **el worker real del repo** (`scripts/service-bus-ollama-worker.ts`), no una reimplementacion.
Colab aporta la GPU; el codigo sale de `eydersantiago/PDC`.

El worker es *pull*: se conecta el a Service Bus. No necesita IP fija, ni puerto abierto,
ni tocar nada en Azure para cambiar de maquina. Convive con otros workers por competing
consumers: cuando este notebook esta vivo se lleva la mayoria de los jobs; cuando lo
cierras, los demas siguen solos.

## Mismo worker en un Mac

```bash
brew install ollama git node
ollama serve &
ollama pull qwen2.5:7b-instruct
git clone -b claude/amazing-fermi-f6a23q https://github.com/eydersantiago/PDC.git
cd PDC && npm ci
cp .env.worker.example .env.worker      # rellenar y poner QUEUE_WORKER_ID=mac-<nombre>
npm run worker:queue
```

Es literalmente lo que hacen las celdas de abajo. En Apple Silicon la memoria unificada
te deja subir de modelo sin cambiar nada mas que `MODEL_TEXT`.

---

**Antes de empezar:** *Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU (T4)*.
Los compute units se queman por tiempo encendido, no por computo: apaga el runtime al terminar.


## 1. Configuracion

La SAS tiene que ser de **namespace** con Listen + Send (politica `colab-worker`).
Una SAS de cola no sirve: el worker abre un solo `ServiceBusClient` y de ahi saca
receptor sobre `llm-jobs` y emisor sobre `llm-results-sessions`. La celda lo verifica.


In [ ]:
from getpass import getpass
import re

REPO_URL = "https://github.com/eydersantiago/PDC.git"
BRANCH   = "claude/amazing-fermi-f6a23q"
REPO_DIR = "/content/PDC"
API_BASE = "https://app-adaceen-api-eyder05232002.azurewebsites.net"

MODEL_TEXT         = "qwen2.5:7b-instruct"
JOBS_QUEUE_NAME    = "llm-jobs"
RESULTS_QUEUE_NAME = "llm-results-sessions"
QUEUE_WORKER_ID    = "colab-t4"

SB_CONN = getpass("SAS namespace Listen+Send (colab-worker): ").strip()
WORKER_SHARED_SECRET = getpass("WORKER_SHARED_SECRET (Enter si no se usa): ").strip()

campo = lambda c, k: (re.search(k + "=([^;]+)", c or "") or [None, ""])[1]
politica = campo(SB_CONN, "SharedAccessKeyName")
entidad  = campo(SB_CONN, "EntityPath")

print("politica  :", politica or "(ausente)")
print("EntityPath:", entidad or "(ninguno) <- correcto, es de namespace")
print("secreto   :", "si" if WORKER_SHARED_SECRET else "no")
assert not entidad, \
    "Esa SAS esta acotada a la cola '%s'. Necesitas una de namespace con Listen+Send." % entidad
assert politica, "La cadena no parece una SAS valida."


## 2. Ollama

`zstd` es obligatorio: el instalador de Ollama empaqueta en `.tar.zst` y la imagen de
Colab no lo trae. Sin el, la instalacion falla y `ollama serve` da `FileNotFoundError`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh -o /tmp/ins.sh && sh /tmp/ins.sh 2>&1 | tail -2
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 10; ollama pull {MODEL_TEXT}
!grep -m1 -i "inference compute" /tmp/ollama.log


## 3. Repo y dependencias

Clona la rama y instala. `npm ci` si hay lockfile, `npm install` si no.
Es la parte lenta del arranque (~2 min); Ollama tarda ~1.


In [ ]:
!node -v >/dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1)
!echo "node $(node -v)  npm $(npm -v)"
!test -d {REPO_DIR}/.git && (cd {REPO_DIR} && git fetch -q --depth 1 origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard FETCH_HEAD) || git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
!cd {REPO_DIR} && git log -1 --oneline
!cd {REPO_DIR} && (npm ci --no-audit --no-fund 2>&1 | tail -3 || npm install --no-audit --no-fund 2>&1 | tail -3)


## 4. `.env.worker`

El worker lee `.env` y luego `.env.worker` con `override: true`. Escribimos solo el segundo.
La celda imprime los **nombres** de las variables, nunca los valores.


In [ ]:
import pathlib

lineas = [
    "AGENT_TARGET=queue",
    "AZURE_SERVICEBUS_CONNECTION_STRING=" + SB_CONN,
    "JOBS_QUEUE_NAME=" + JOBS_QUEUE_NAME,
    "RESULTS_QUEUE_NAME=" + RESULTS_QUEUE_NAME,
    "WORKER_SHARED_SECRET=" + WORKER_SHARED_SECRET,
    "QUEUE_REQUEST_TIMEOUT_MS=120000",
    "QUEUE_WORKER_ID=" + QUEUE_WORKER_ID,
    "QUEUE_WORKER_MAX_ATTEMPTS=3",
    "QUEUE_WORKER_RETRY_DELAY_MS=2000",
    "QUEUE_WORKER_LOCK_RENEWAL_MS=0",
    "OPENAI_BASE=http://127.0.0.1:11434/v1",
    "OPENAI_API_KEY=dummy",
    "OLLAMA_BASE_URL=http://127.0.0.1:11434",
    "MODEL_TEXT=" + MODEL_TEXT,
    "ADACEEN_LOG_LEVEL=info",
    "ADACEEN_LOG_STACKS=0",
]

ruta = pathlib.Path(REPO_DIR) / ".env.worker"
ruta.write_text(chr(10).join(lineas) + chr(10))
print(ruta, "->", len(lineas), "variables")
print("  " + "  ".join(l.split("=")[0] for l in lineas))

gi = pathlib.Path(REPO_DIR, ".gitignore")
texto = gi.read_text() if gi.exists() else ""
ignorado = any(p in texto for p in (".env.worker", ".env*", ".env"))
print("gitignored:", ignorado, "" if ignorado else "<- OJO: no lo commitees")


## 5. Arrancar el worker

En segundo plano, para que el notebook quede libre y puedas lanzar pruebas desde
otra celda. El log va a `/content/worker.log`.


In [ ]:
!cd {REPO_DIR} && nohup npm run worker:queue > /content/worker.log 2>&1 &
!sleep 20; tail -30 /content/worker.log


## 6. Prueba end-to-end

Lanza una peticion real contra el API. El API publica en `llm-jobs`, este worker la
consume, responde en `llm-results-sessions` y el API te devuelve el texto.
Si la cola estaba vacia, esto es lo que la llena.


In [ ]:
import requests, time

payload = {"input_as_text": "def duplicar(n): return n * 2"}

for ruta_api in ("/run-text", "/api/run-text"):
    t0 = time.time()
    try:
        r = requests.post(API_BASE + ruta_api, json=payload, timeout=180)
    except Exception as e:
        print(ruta_api, "->", type(e).__name__, e)
        continue
    print(ruta_api, "-> HTTP", r.status_code, "en %.1fs" % (time.time() - t0))
    if r.status_code != 404:
        print(r.text[:2000])
        break


## 7. Log y parada


In [ ]:
!tail -40 /content/worker.log
# !pkill -f service-bus-ollama-worker    # detener el worker
# !cd {REPO_DIR} && npm run worker:queue  # o correrlo en primer plano para ver todo
